**Architecture**-

Driver : (brain of the operation) It runs your main PySpark code, creates the SparkSession, and decides how to split up the work.

Cluster Manager: The resource allocator grabs necessary RAM and CPU power.

Executors: These are the workers that execute the tasks assigned by the Driver and store data in their memory

**Lazy Evaluation and DAG**

When you apply Transformations (like filter ,withColumn), Spark does not execute them immediately.it builds a DAG (Directed Acyclic Graph), which is a step-by-step blueprint or lineage of everything you want to do.

Spark only executes the blueprint when call an Action. This allows Spark to look at the whole DAG and optimize the execution plan before doing any real work.


CODE STARTS FROM **HERE**

In [6]:

!pip install pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("week_6_pipeline") \
    .getOrCreate()



In [7]:

path = "/content/jeans.csv"
df = spark.read.csv(path, header=True, inferSchema=True)


print("5 Rows of the Jeans Dataset")
df.show(5)  #its an Ation

# printing Schema
print("Data Schema (Column Types)")
df.printSchema()

5 Rows of the Jeans Dataset
+----------+------------------+--------------------+------+-------------+-------------+--------+-------------+---------+--------------------+--------------------+--------------------+--------------------+----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|product_id|             title| product_description|rating|ratings_count|initial_price|discount|  final_price| currency|              images|    delivery_options|     product_details|         breadcrumbs|product_specifications|     amount_of_stars| what_customers_said|         seller_name|               sizes|              videos|  seller_information|          variations|          best_offer|         more_offers|            category|
+----------+------------------+--------------------+------+-------------+-------------+--------+

Modify & Transform

In [8]:
from pyspark.sql.functions import col, round

# Keeping only clean columns
selected_df = df.select("product_id", "title", "rating", "initial_price", "discount")

# changing column name
transformed_df = selected_df.withColumnRenamed("title", "brand_name")

# 3. decimal to whole number using cast
transformed_df = transformed_df.withColumn("discount", col("discount").cast("integer"))

# calculating final price
transformed_df = transformed_df.withColumn(
    "calculated_finalPrice",
    round(col("initial_price") * (1 - (col("discount") / 100)), 2)
)


print("Transformed DataFrame")
transformed_df.show(5)

print("New Schema ")
transformed_df.printSchema()

Transformed DataFrame
+----------+------------------+------+-------------+--------+---------------------+
|product_id|        brand_name|rating|initial_price|discount|calculated_finalPrice|
+----------+------------------+------+-------------+--------+---------------------+
|  18973692|             Levis|   4.2|       4099.0|      57|              1762.57|
|  18973742|             Levis|   4.2|       4399.0|      48|              2287.48|
|  19008522|Calvin Klein Jeans|   3.8|       7999.0|      20|               6399.2|
|  18974084|             Levis|   4.2|       4399.0|      57|              1891.57|
|  18873126|             KETCH|   4.0|       1749.0|      64|               629.64|
+----------+------------------+------+-------------+--------+---------------------+
only showing top 5 rows
New Schema 
root
 |-- product_id: integer (nullable = true)
 |-- brand_name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- initial_price: double (nullable = true)
 |-- discount

**Null Values and Filter Data**

In [9]:
# dropping null value records
cleaned_df = transformed_df.dropna(subset=["brand_name", "calculated_finalPrice"])

# filtering
filtered_df = cleaned_df.filter(col("rating") >= 4.0).filter(col("discount") > 0)

print("Cleaned & Filtered Data")
filtered_df.show(5)

print(f"Total rows after filtering:{filtered_df.count()}")

Cleaned & Filtered Data
+----------+--------------------+------+-------------+--------+---------------------+
|product_id|          brand_name|rating|initial_price|discount|calculated_finalPrice|
+----------+--------------------+------+-------------+--------+---------------------+
|  18973692|               Levis|   4.2|       4099.0|      57|              1762.57|
|  18973742|               Levis|   4.2|       4399.0|      48|              2287.48|
|  18974084|               Levis|   4.2|       4399.0|      57|              1891.57|
|  18873126|               KETCH|   4.0|       1749.0|      64|               629.64|
|  18802522|BEAT LONDON by PE...|   4.0|       2799.0|      50|               1399.5|
+----------+--------------------+------+-------------+--------+---------------------+
only showing top 5 rows
Total rows after filtering:29


In [10]:
# saving
filtered_df.write.mode("overwrite").csv("/content/jeans_cleaned_csv", header=True)

# 2. Save the processed data as a Parquet file
filtered_df.write.mode("overwrite").parquet("/content/jeans_cleaned_parquet")

print("Files saved in both formats.")

Files saved in both formats.


**Performance & Best Practices**

CSV (Row-based): Good for human readability, but not for big data performance. It takes up a lot of storage space, is slow to read/write, and completely forgets schema when saved.

Parquet (Column-based): This is preferred format. It highly compresses the data , remembers the exact schema, and uses "Predicate Pushdown" (meaning if you only query one column, Spark doesn't have to read whole file, making it very fast).

**Best Practices for Large Datasets**

Avoid .collect(): This command attempts to grab all the data from the Worker nodes and put it into the single Driver node's memory. On a large dataset, this will crash the application with an error.

Use .show(): Throughout this pipeline, I used .show(5) to safely preview the data without overwhelming the Driver.